# Gold Layer — Star Schema Salutar Saúde

## Modelo Dimensional

```
           ┌──────────────────┐
           │  dim_operadora   │
           │  PK: operadora_id│◄──────────────────────┐
           └──────────────────┘                       │
                                                      │
┌─────────────────┐  ┌──────────────────┐             │  ┌──────────────────┐
│  dim_empresa    │  │    dim_plano      │─────────────┘  │  dim_corretor    │
│  PK: empresa_id │  │  PK: plano_id    │                │  PK: corretor_id │
└────────┬────────┘  └────────┬─────────┘                └────────┬─────────┘
         │                   │                                    │
         │  ┌────────────────▼────────────────────────────────▼──────────┐
         ├─►│                    fat_contratos                                │◄── dim_data
         │  │  PK: contrato_id                                                │    (vigências)
         │  │  FK: empresa_id · plano_id · corretor_id                        │
         │  │  Méd: num_vidas · valor_mensal · receita_total_mensal           │
         │  └────────────────┬──────────────────────────────────────────────┘
         │                   │
         │  ┌────────────────▼────────────────┐  ┌──────────────────────┐
         │  │        dim_beneficiario            │  │       dim_data       │
         │  │  PK: beneficiario_id               │  │  PK: data (DATE)     │
         │  │  FK: contrato_id · empresa_id      │  │      data_id (INT)   │
         │  └──────────────────┬────────────────┘  └──────────┬───────────┘
         │                     ▲                               │
         │    ┌────────────────────┬──────────────────────────▼────────────┐
         ├───►│                    fat_utilizacao                            │
         │    │  PK: evento_id                                               │
         │    │  FK: beneficiario_id · contrato_id · empresa_id · plano_id  │
         │    │  Méd: valor_sinistro                                         │
         │    └────────────────────────────────────────────────────────┘
         │
         │  ┌─────────────────────────────────────────────────────────────────┐
         └─►│                   fat_sinistralidade                          │◄── dim_data
            │  PK: empresa_id × ano_mes                                     │    (via ano_mes)
            │  FK: empresa_id → dim_empresa · ano_mes → dim_data            │
            │  Méd: sinistralidade_pct · receita_premios · custo_sinistros  │
            │       margem_tecnica · custo_per_capita                       │
            └─────────────────────────────────────────────────────────────────┘
```

## Grãos e Volumes

| Tabela | Grão | Volume (aprox.) |
|---|---|---|
| `dim_operadora` | 1 linha por operadora | ~8 |
| `dim_empresa` | 1 linha por empresa | ~41 |
| `dim_plano` | 1 linha por plano | ~28 |
| `dim_corretor` | 1 linha por corretor | ~25 |
| `dim_data` | 1 linha por dia do calendário | ~4.748 (2018–2030) |
| `dim_beneficiario` | 1 linha por beneficiário | ~1.558 |
| `fat_contratos` | 1 linha por contrato | ~109 |
| `fat_utilizacao` | 1 linha por evento de utilização (sinistro) | ~6.465 |
| `fat_sinistralidade` | 1 linha por empresa × mês (pré-agregada) | ~empresa × meses ativos |

## Conexões de dim_data

`dim_data` é uma **dimensão conformada** — múltiplas colunas de data das facts fazem JOIN nela usando a coluna `data` (DATE):

| Fact | Coluna | Expressão de JOIN |
|---|---|---|
| `fat_contratos` | `data_venda` | `fat_contratos.data_venda = dim_data.data` |
| `fat_contratos` | `vigencia_inicio` | `fat_contratos.vigencia_inicio = dim_data.data` |
| `fat_contratos` | `vigencia_fim` | `fat_contratos.vigencia_fim = dim_data.data` |
| `fat_utilizacao` | `data_evento` | `fat_utilizacao.data_evento = dim_data.data` |
| `fat_sinistralidade` | `ano_mes` | `fat_sinistralidade.ano_mes = dim_data.ano_mes` |

In [0]:
from datetime import datetime

CATALOG = "homologacao"
SCHEMA  = "salutar_saude"

SILVER = f"{CATALOG}.{SCHEMA}"
GOLD   = f"{CATALOG}.{SCHEMA}"

run_ts = datetime.now()
print(f"Gold Star Schema — Salutar Saúde")
print(f"Iniciado : {run_ts:%Y-%m-%d %H:%M:%S}")
print(f"Origem   : {SILVER}.silver_*")
print(f"Destino  : {GOLD}.gold_*")

Gold Star Schema — Salutar Saúde
Iniciado : 2026-07-04 20:55:39
Origem   : homologacao.salutar_saude.silver_*
Destino  : homologacao.salutar_saude.gold_*


In [0]:
%sql
-- =============================================================================
-- dim_operadora
-- Grão  : 1 linha por operadora de saúde
-- Fonte : silver_operadoras
-- PK    : operadora_id
-- =============================================================================
CREATE OR REPLACE TABLE homologacao.salutar_saude.gold_dim_operadora
COMMENT 'Dimensão de operadoras de saúde. Grão: 1 linha por operadora. Fonte: silver_operadoras.'
AS
SELECT
  operadora_id,   -- PK
  operadora_nome
FROM homologacao.salutar_saude.silver_operadoras;

-- Comentários de colunas
ALTER TABLE homologacao.salutar_saude.gold_dim_operadora
  ALTER COLUMN operadora_id COMMENT 'Chave primária da operadora';
ALTER TABLE homologacao.salutar_saude.gold_dim_operadora
  ALTER COLUMN operadora_nome COMMENT 'Nome comercial da operadora';

SELECT COUNT(*) AS total_operadoras FROM homologacao.salutar_saude.gold_dim_operadora;

total_operadoras
8


In [0]:
%sql
-- =============================================================================
-- dim_empresa
-- Grão  : 1 linha por empresa contratante
-- Fonte : silver_empresas
-- PK    : empresa_id
-- =============================================================================
CREATE OR REPLACE TABLE homologacao.salutar_saude.gold_dim_empresa
COMMENT 'Dimensão de empresas contratantes. Grão: 1 linha por empresa. Fonte: silver_empresas.'
AS
SELECT
  empresa_id,                                              -- PK
  empresa_nome,
  setor,
  porte,
  uf,
  CAST(data_inicio_relacionamento AS DATE) AS data_inicio_relacionamento
FROM homologacao.salutar_saude.silver_empresas;

ALTER TABLE homologacao.salutar_saude.gold_dim_empresa
  ALTER COLUMN empresa_id COMMENT 'Chave primária da empresa';
ALTER TABLE homologacao.salutar_saude.gold_dim_empresa
  ALTER COLUMN setor COMMENT 'Setor econômico da empresa (ex.: Saúde, Tecnologia, Varejo)';
ALTER TABLE homologacao.salutar_saude.gold_dim_empresa
  ALTER COLUMN porte COMMENT 'Porte da empresa: Pequena | Média | Grande';
ALTER TABLE homologacao.salutar_saude.gold_dim_empresa
  ALTER COLUMN uf COMMENT 'Sigla do estado (2 letras)';
ALTER TABLE homologacao.salutar_saude.gold_dim_empresa
  ALTER COLUMN data_inicio_relacionamento COMMENT 'Data de início do relacionamento comercial';

SELECT COUNT(*) AS total_empresas FROM homologacao.salutar_saude.gold_dim_empresa;

total_empresas
40


In [0]:
%sql
-- =============================================================================
-- dim_plano
-- Grão  : 1 linha por plano de saúde
-- Fonte : silver_planos
-- PK    : plano_id
-- FK    : operadora_id → gold_dim_operadora
-- =============================================================================
CREATE OR REPLACE TABLE homologacao.salutar_saude.gold_dim_plano
COMMENT 'Dimensão de planos de saúde. Grão: 1 linha por plano. FK: operadora_id → gold_dim_operadora. Fonte: silver_planos.'
AS
SELECT
  plano_id,       -- PK
  operadora_id,   -- FK → gold_dim_operadora
  plano_nome,
  segmentacao,
  acomodacao,
  coparticipacao,
  preco_vida_mes
FROM homologacao.salutar_saude.silver_planos;

ALTER TABLE homologacao.salutar_saude.gold_dim_plano
  ALTER COLUMN plano_id COMMENT 'Chave primária do plano';
ALTER TABLE homologacao.salutar_saude.gold_dim_plano
  ALTER COLUMN operadora_id COMMENT 'FK → gold_dim_operadora';
ALTER TABLE homologacao.salutar_saude.gold_dim_plano
  ALTER COLUMN segmentacao COMMENT 'Tipo de cobertura: Ambulatorial | Hospitalar | Ambulatorial + Hospitalar | Hospitalar + Obstetrícia';
ALTER TABLE homologacao.salutar_saude.gold_dim_plano
  ALTER COLUMN acomodacao COMMENT 'Tipo de acomodação: Enfermaria | Apartamento';
ALTER TABLE homologacao.salutar_saude.gold_dim_plano
  ALTER COLUMN coparticipacao COMMENT 'Possui coparticipação: Sim | Não';
ALTER TABLE homologacao.salutar_saude.gold_dim_plano
  ALTER COLUMN preco_vida_mes COMMENT 'Preço por vida por mês em R$';

SELECT COUNT(*) AS total_planos FROM homologacao.salutar_saude.gold_dim_plano;

total_planos
28


In [0]:
%sql
-- =============================================================================
-- dim_corretor
-- Grão  : 1 linha por corretor
-- Fonte : silver_corretores
-- PK    : corretor_id
-- =============================================================================
CREATE OR REPLACE TABLE homologacao.salutar_saude.gold_dim_corretor
COMMENT 'Dimensão de corretores de venda. Grão: 1 linha por corretor. Fonte: silver_corretores.'
AS
SELECT
  corretor_id,   -- PK
  corretor_nome,
  regiao,
  senioridade,
  CAST(data_admissao AS DATE) AS data_admissao
FROM homologacao.salutar_saude.silver_corretores;

ALTER TABLE homologacao.salutar_saude.gold_dim_corretor
  ALTER COLUMN corretor_id COMMENT 'Chave primária do corretor';
ALTER TABLE homologacao.salutar_saude.gold_dim_corretor
  ALTER COLUMN regiao COMMENT 'Região de atuação: Norte | Nordeste | Centro-Oeste | Sudeste | Sul';
ALTER TABLE homologacao.salutar_saude.gold_dim_corretor
  ALTER COLUMN senioridade COMMENT 'Nível de senioridade: Júnior | Pleno | Sênior';
ALTER TABLE homologacao.salutar_saude.gold_dim_corretor
  ALTER COLUMN data_admissao COMMENT 'Data de admissão do corretor';

SELECT COUNT(*) AS total_corretores FROM homologacao.salutar_saude.gold_dim_corretor;

total_corretores
25


In [0]:
%sql
-- =============================================================================
-- dim_data  (dimensão de datas — calendário)
-- Grão  : 1 linha por dia do calendário
-- Range : 2018-01-01 → 2030-12-31  (~4.748 linhas)
-- PK    : data (DATE)   — use para JOIN direto com as facts
--         data_id (INT) — chave inteira YYYYMMDD, opcional para ferramentas de BI
-- Conecta-se a:
--   fat_contratos  : data_venda · vigencia_inicio · vigencia_fim
--   fat_utilizacao : data_evento
-- =============================================================================
CREATE OR REPLACE TABLE homologacao.salutar_saude.gold_dim_data
COMMENT 'Dimensão de datas (calendário). Grão: 1 linha por dia. Range: 2018-01-01 a 2030-12-31. PK natural: data (DATE). PK inteira: data_id (YYYYMMDD).'
AS
WITH calendario AS (
  SELECT explode(sequence(DATE '2018-01-01', DATE '2030-12-31', INTERVAL 1 DAY)) AS data
)
SELECT
  CAST(date_format(data, 'yyyyMMdd') AS INT)           AS data_id,        -- PK inteira  (ex.: 20240315)
  data,                                                                    -- PK natural  (DATE)
  YEAR(data)                                           AS ano,
  QUARTER(data)                                        AS trimestre,       -- 1 a 4
  CASE WHEN MONTH(data) <= 6 THEN 1 ELSE 2 END         AS semestre,        -- 1 (Jan-Jun) | 2 (Jul-Dez)
  MONTH(data)                                          AS mes,             -- 1 a 12
  DAYOFMONTH(data)                                     AS dia,
  WEEKOFYEAR(data)                                     AS semana_ano,      -- ISO week 1-53
  DAYOFWEEK(data)                                      AS dia_semana,      -- 1=Dom … 7=Sáb
  CASE DAYOFWEEK(data)
    WHEN 1 THEN 'Domingo'        WHEN 2 THEN 'Segunda-feira'
    WHEN 3 THEN 'Terça-feira'    WHEN 4 THEN 'Quarta-feira'
    WHEN 5 THEN 'Quinta-feira'   WHEN 6 THEN 'Sexta-feira'
    WHEN 7 THEN 'Sábado'
  END                                                  AS nome_dia_semana,
  CASE MONTH(data)
    WHEN 1  THEN 'Janeiro'      WHEN 2  THEN 'Fevereiro'
    WHEN 3  THEN 'Março'        WHEN 4  THEN 'Abril'
    WHEN 5  THEN 'Maio'         WHEN 6  THEN 'Junho'
    WHEN 7  THEN 'Julho'        WHEN 8  THEN 'Agosto'
    WHEN 9  THEN 'Setembro'     WHEN 10 THEN 'Outubro'
    WHEN 11 THEN 'Novembro'     WHEN 12 THEN 'Dezembro'
  END                                                  AS nome_mes,
  date_format(data, 'yyyy-MM')                         AS ano_mes,         -- ex.: '2024-03'
  date_format(data, 'yyyy-QQ')                         AS ano_trimestre,   -- ex.: '2024-Q2'
  CASE WHEN DAYOFWEEK(data) IN (1, 7) THEN TRUE ELSE FALSE END
                                                       AS is_fim_semana,
  LAST_DAY(data)                                       AS ultimo_dia_mes
FROM calendario
ORDER BY data;

ALTER TABLE homologacao.salutar_saude.gold_dim_data
  ALTER COLUMN data_id        COMMENT 'PK inteira no formato YYYYMMDD (ex.: 20240315)';
ALTER TABLE homologacao.salutar_saude.gold_dim_data
  ALTER COLUMN data           COMMENT 'PK natural (DATE); use para JOIN com fat_contratos.data_venda, fat_utilizacao.data_evento etc.';
ALTER TABLE homologacao.salutar_saude.gold_dim_data
  ALTER COLUMN trimestre      COMMENT 'Trimestre do ano: 1 a 4';
ALTER TABLE homologacao.salutar_saude.gold_dim_data
  ALTER COLUMN semestre       COMMENT '1 = Janeiro a Junho | 2 = Julho a Dezembro';
ALTER TABLE homologacao.salutar_saude.gold_dim_data
  ALTER COLUMN dia_semana     COMMENT '1=Domingo, 2=Segunda-feira, 3=Terça-feira, 4=Quarta-feira, 5=Quinta-feira, 6=Sexta-feira, 7=Sábado';
ALTER TABLE homologacao.salutar_saude.gold_dim_data
  ALTER COLUMN ano_mes        COMMENT 'Período no formato YYYY-MM (ex.: 2024-03); útil para GROUP BY mensal';
ALTER TABLE homologacao.salutar_saude.gold_dim_data
  ALTER COLUMN ano_trimestre  COMMENT 'Período no formato YYYY-QN (ex.: 2024-Q2); útil para GROUP BY trimestral';
ALTER TABLE homologacao.salutar_saude.gold_dim_data
  ALTER COLUMN is_fim_semana  COMMENT 'TRUE se o dia for Sábado (7) ou Domingo (1)';
ALTER TABLE homologacao.salutar_saude.gold_dim_data
  ALTER COLUMN ultimo_dia_mes COMMENT 'Último dia do mês (DATE); útil para filtros de fechamento mensal';

SELECT
  MIN(data)    AS data_min,
  MAX(data)    AS data_max,
  COUNT(*)     AS total_dias
FROM homologacao.salutar_saude.gold_dim_data;

data_min,data_max,total_dias
2018-01-01,2030-12-31,4748


In [0]:
%sql
-- =============================================================================
-- dim_beneficiario
-- Grão  : 1 linha por beneficiário
-- Fonte : silver_beneficiarios
-- PK    : beneficiario_id
-- FK    : contrato_id → gold_fat_contratos
--         empresa_id  → gold_dim_empresa
-- Nota  : data_cancelamento é NULL para beneficiários ativos (~89% dos registros)
-- =============================================================================
CREATE OR REPLACE TABLE homologacao.salutar_saude.gold_dim_beneficiario
COMMENT 'Dimensão de beneficiários. Grão: 1 linha por beneficiário. FKs: contrato_id, empresa_id. Fonte: silver_beneficiarios.'
AS
SELECT
  beneficiario_id,                                           -- PK
  contrato_id,                                               -- FK → gold_fat_contratos
  empresa_id,                                                -- FK → gold_dim_empresa
  CAST(data_nascimento AS DATE)   AS data_nascimento,
  sexo,
  tipo_beneficiario,
  CAST(data_adesao AS DATE)       AS data_adesao,
  CAST(data_cancelamento AS DATE) AS data_cancelamento,
  -- campo derivado: situação atual do beneficiário
  CASE
    WHEN data_cancelamento IS NOT NULL THEN 'Cancelado'
    ELSE 'Ativo'
  END AS situacao,
  -- campo derivado: faixa etária (classificador comum em saúde)
  CASE
    WHEN DATEDIFF(CURRENT_DATE(), CAST(data_nascimento AS DATE)) / 365.25 < 18  THEN '00-17'
    WHEN DATEDIFF(CURRENT_DATE(), CAST(data_nascimento AS DATE)) / 365.25 < 30  THEN '18-29'
    WHEN DATEDIFF(CURRENT_DATE(), CAST(data_nascimento AS DATE)) / 365.25 < 45  THEN '30-44'
    WHEN DATEDIFF(CURRENT_DATE(), CAST(data_nascimento AS DATE)) / 365.25 < 60  THEN '45-59'
    ELSE '60+'
  END AS faixa_etaria
FROM homologacao.salutar_saude.silver_beneficiarios;

ALTER TABLE homologacao.salutar_saude.gold_dim_beneficiario
  ALTER COLUMN beneficiario_id COMMENT 'Chave primária do beneficiário';
ALTER TABLE homologacao.salutar_saude.gold_dim_beneficiario
  ALTER COLUMN contrato_id COMMENT 'FK → gold_fat_contratos';
ALTER TABLE homologacao.salutar_saude.gold_dim_beneficiario
  ALTER COLUMN empresa_id COMMENT 'FK → gold_dim_empresa';
ALTER TABLE homologacao.salutar_saude.gold_dim_beneficiario
  ALTER COLUMN tipo_beneficiario COMMENT 'Titular ou Dependente';
ALTER TABLE homologacao.salutar_saude.gold_dim_beneficiario
  ALTER COLUMN data_cancelamento COMMENT 'Data de cancelamento; NULL = beneficiário ativo';
ALTER TABLE homologacao.salutar_saude.gold_dim_beneficiario
  ALTER COLUMN situacao COMMENT 'Ativo (data_cancelamento IS NULL) | Cancelado';
ALTER TABLE homologacao.salutar_saude.gold_dim_beneficiario
  ALTER COLUMN faixa_etaria COMMENT 'Faixa etária calculada com base em data_nascimento: 00-17 | 18-29 | 30-44 | 45-59 | 60+';

SELECT COUNT(*) AS total_beneficiarios FROM homologacao.salutar_saude.gold_dim_beneficiario;

total_beneficiarios
1558


In [0]:
%sql
-- =============================================================================
-- fat_contratos
-- Grão  : 1 linha por contrato de plano de saúde
-- Fonte : silver_contratos
-- PK    : contrato_id
-- FK    : empresa_id   → gold_dim_empresa
--         plano_id     → gold_dim_plano
--         corretor_id  → gold_dim_corretor
-- Métricas: num_vidas, valor_mensal, receita_total_mensal, dias_vigencia
-- =============================================================================
CREATE OR REPLACE TABLE homologacao.salutar_saude.gold_fat_contratos
COMMENT 'Fato de contratos de plano de saúde. Grão: 1 linha por contrato. FKs: empresa_id, plano_id, corretor_id. Fonte: silver_contratos.'
AS
SELECT
  contrato_id,                                              -- PK
  empresa_id,                                               -- FK → gold_dim_empresa
  plano_id,                                                 -- FK → gold_dim_plano
  corretor_id,                                              -- FK → gold_dim_corretor
  CAST(data_venda       AS DATE) AS data_venda,
  CAST(vigencia_inicio  AS DATE) AS vigencia_inicio,
  CAST(vigencia_fim     AS DATE) AS vigencia_fim,
  num_vidas,
  valor_mensal,
  status,
  -- métricas derivadas
  COALESCE(valor_mensal * num_vidas, 0) AS receita_total_mensal,
  DATEDIFF(
    CAST(vigencia_fim    AS DATE),
    CAST(vigencia_inicio AS DATE)
  ) AS dias_vigencia
FROM homologacao.salutar_saude.silver_contratos;

ALTER TABLE homologacao.salutar_saude.gold_fat_contratos
  ALTER COLUMN contrato_id COMMENT 'Chave primária do contrato';
ALTER TABLE homologacao.salutar_saude.gold_fat_contratos
  ALTER COLUMN empresa_id COMMENT 'FK → gold_dim_empresa';
ALTER TABLE homologacao.salutar_saude.gold_fat_contratos
  ALTER COLUMN plano_id COMMENT 'FK → gold_dim_plano';
ALTER TABLE homologacao.salutar_saude.gold_fat_contratos
  ALTER COLUMN corretor_id COMMENT 'FK → gold_dim_corretor';
ALTER TABLE homologacao.salutar_saude.gold_fat_contratos
  ALTER COLUMN num_vidas COMMENT 'Quantidade de vidas cobertas pelo contrato';
ALTER TABLE homologacao.salutar_saude.gold_fat_contratos
  ALTER COLUMN valor_mensal COMMENT 'Valor mensal total do contrato em R$';
ALTER TABLE homologacao.salutar_saude.gold_fat_contratos
  ALTER COLUMN status COMMENT 'Situação do contrato: Ativo | Cancelado | Renovado';
ALTER TABLE homologacao.salutar_saude.gold_fat_contratos
  ALTER COLUMN receita_total_mensal COMMENT 'valor_mensal * num_vidas (métrica derivada)';
ALTER TABLE homologacao.salutar_saude.gold_fat_contratos
  ALTER COLUMN dias_vigencia COMMENT 'Duração em dias entre vigencia_inicio e vigencia_fim';

SELECT COUNT(*) AS total_contratos FROM homologacao.salutar_saude.gold_fat_contratos;

total_contratos
108


In [0]:
%sql
-- =============================================================================
-- fat_utilizacao
-- Grão  : 1 linha por evento de utilização (sinistro)
-- Fonte : silver_utilizacao + silver_beneficiarios + silver_contratos
-- PK    : evento_id
-- FK    : beneficiario_id  → gold_dim_beneficiario
--         contrato_id      → gold_fat_contratos     (derivado via beneficiario)
--         empresa_id       → gold_dim_empresa       (derivado via beneficiario)
--         plano_id         → gold_dim_plano         (derivado via contrato)
-- Métrica: valor_sinistro
-- =============================================================================
CREATE OR REPLACE TABLE homologacao.salutar_saude.gold_fat_utilizacao
COMMENT 'Fato de eventos de utilização (sinistros). Grão: 1 linha por evento. FKs: beneficiario_id, contrato_id, empresa_id, plano_id. Fonte: silver_utilizacao + silver_beneficiarios + silver_contratos.'
AS
SELECT
  u.evento_id,                                              -- PK
  u.beneficiario_id,                                        -- FK → gold_dim_beneficiario
  b.contrato_id,                                            -- FK → gold_fat_contratos
  b.empresa_id,                                             -- FK → gold_dim_empresa
  c.plano_id,                                               -- FK → gold_dim_plano
  CAST(u.data_evento AS DATE) AS data_evento,
  u.tipo_evento,
  u.especialidade,
  u.valor_sinistro
FROM      homologacao.salutar_saude.silver_utilizacao    u
INNER JOIN homologacao.salutar_saude.silver_beneficiarios b ON u.beneficiario_id = b.beneficiario_id
INNER JOIN homologacao.salutar_saude.silver_contratos     c ON b.contrato_id     = c.contrato_id
-- Dedup defensivo: duplicata em silver_contratos causa fan-out no JOIN → remove linhas extras
QUALIFY ROW_NUMBER() OVER (PARTITION BY u.evento_id ORDER BY u.evento_id) = 1;

ALTER TABLE homologacao.salutar_saude.gold_fat_utilizacao
  ALTER COLUMN evento_id COMMENT 'Chave primária do evento de utilização';
ALTER TABLE homologacao.salutar_saude.gold_fat_utilizacao
  ALTER COLUMN beneficiario_id COMMENT 'FK → gold_dim_beneficiario';
ALTER TABLE homologacao.salutar_saude.gold_fat_utilizacao
  ALTER COLUMN contrato_id COMMENT 'FK → gold_fat_contratos (derivado via beneficiario_id)';
ALTER TABLE homologacao.salutar_saude.gold_fat_utilizacao
  ALTER COLUMN empresa_id COMMENT 'FK → gold_dim_empresa (derivado via beneficiario_id)';
ALTER TABLE homologacao.salutar_saude.gold_fat_utilizacao
  ALTER COLUMN plano_id COMMENT 'FK → gold_dim_plano (derivado via contrato_id)';
ALTER TABLE homologacao.salutar_saude.gold_fat_utilizacao
  ALTER COLUMN tipo_evento COMMENT 'Tipo do evento: Consulta | Exame | Cirurgia | Pronto-socorro | Internação | ...';
ALTER TABLE homologacao.salutar_saude.gold_fat_utilizacao
  ALTER COLUMN especialidade COMMENT 'Especialidade médica do evento (ex.: Cardiologia, Oncologia)';
ALTER TABLE homologacao.salutar_saude.gold_fat_utilizacao
  ALTER COLUMN valor_sinistro COMMENT 'Valor do sinistro em R$ (custo do evento)';

SELECT COUNT(*) AS total_eventos FROM homologacao.salutar_saude.gold_fat_utilizacao;

total_eventos
6450


In [0]:
%sql
-- =============================================================================
-- fat_sinistralidade
-- Grão  : 1 linha por empresa × ano_mes
-- PK    : (empresa_id, ano_mes)  — chave composta
-- Fonte : gold_fat_contratos + gold_fat_utilizacao + gold_dim_empresa
-- Métrica principal : sinistralidade = custo_sinistros / receita_premios
-- =============================================================================
-- ── PREMISSAS DE CÁLCULO ─────────────────────────────────────────────────────────────────────
--
-- [P1] RECEITA DE PRÊMLIOS (denominador)
--      Fonte  : gold_fat_contratos.valor_mensal
--      Regra  : Cada contrato contribui com valor_mensal INTEGRAL em cada mês
--               calendário em que sua vigência contém pelo menos 1 dia.
--      Contratos parciais (início ou fim no meio do mês): receita INTEGRAL,
--               sem pro-rata. Isso SUBestima a sinistralidade nos meses de
--               início/encerramento do contrato.
--      Exclusão : contratos com valor_mensal IS NULL (1 contrato identificado
--               na camada silver) são excluídos do denominador.
--
-- [P2] CUSTO DE SINISTROS (numerador)
--      Fonte  : gold_fat_utilizacao.valor_sinistro
--      Critério de competência: data_evento (data em que o evento ocorreu).
--      Inclui : Consulta · Exame · Cirurgia · Pronto-socorro · Internação e
--               demais tipos registrados (lista aberta).
--      Não inclui : reserva IBNR (ocorridos e não reportados).
--
-- [P3] MÊS COM RECEITA, SEM SINISTRO
--      custo_sinistros = 0  → sinistralidade = 0.0000 (0,00%)
--
-- [P4] MÊS COM SINISTRO, SEM RECEITA
--      receita_premios = 0  → sinistralidade = NULL
--      flag sinistro_sem_receita = TRUE → requer investigação (possível
--      evento de beneficiário com vigência encerrada / cancelamento retroativo).
--
-- [P5] JANELA TEMPORAL
--      Dinâmica: gerada pela união dos mêses de vigência dos contratos e dos
--      mêses com eventos de sinistro. Não há corte fixo de data.
-- =============================================================================
CREATE OR REPLACE TABLE homologacao.salutar_saude.gold_fat_sinistralidade
COMMENT 'Fato de sinistralidade mensal por empresa. Grão: 1 linha por empresa × ano_mes. Sinistralidade = custo_sinistros / receita_premios. Premissas: [P1] receita integral por mês de vigência (sem pro-rata, valor_mensal NULL excluído); [P2] custo por data_evento, sem IBNR; [P3] sem sinistro → sinistralidade=0; [P4] sinistro sem receita → NULL + flag.'
AS

-- ── CTE 1: expande cada contrato nos meses de sua vigência [P1] ─────────────
WITH contratos_por_mes AS (
  SELECT
    c.empresa_id,
    date_format(t.mes, 'yyyy-MM')     AS ano_mes,
    COALESCE(c.valor_mensal, 0)       AS valor_mensal,
    COALESCE(c.num_vidas,   0)        AS num_vidas,
    c.contrato_id
  FROM homologacao.salutar_saude.gold_fat_contratos c
  LATERAL VIEW explode(
    sequence(
      date_trunc('MONTH', c.vigencia_inicio),
      date_trunc('MONTH', c.vigencia_fim),
      INTERVAL 1 MONTH
    )
  ) t AS mes
  WHERE c.valor_mensal IS NOT NULL           -- [P1] exclui contratos sem valor
),

receita_mensal AS (
  SELECT
    empresa_id,
    ano_mes,
    ROUND(SUM(valor_mensal), 2)              AS receita_premios,
    SUM(num_vidas)                           AS vidas_ativas,
    COUNT(DISTINCT contrato_id)              AS contratos_ativos
  FROM contratos_por_mes
  GROUP BY empresa_id, ano_mes
),

-- ── CTE 2: agrega sinistros por empresa × mês do evento [P2] ─────────────
sinistros_por_mes AS (
  SELECT
    empresa_id,
    date_format(data_evento, 'yyyy-MM')      AS ano_mes,
    ROUND(SUM(valor_sinistro), 2)            AS custo_sinistros,
    COUNT(*)                                  AS qtd_eventos
  FROM homologacao.salutar_saude.gold_fat_utilizacao
  GROUP BY empresa_id, date_format(data_evento, 'yyyy-MM')
),

-- ── CTE 3: FULL OUTER JOIN para capturar mêses com receita OU sinistro ──
periodos AS (
  SELECT
    COALESCE(r.empresa_id,       s.empresa_id)  AS empresa_id,
    COALESCE(r.ano_mes,          s.ano_mes)      AS ano_mes,
    COALESCE(r.receita_premios,  0)              AS receita_premios,
    COALESCE(r.vidas_ativas,     0)              AS vidas_ativas,
    COALESCE(r.contratos_ativos, 0)              AS contratos_ativos,
    COALESCE(s.custo_sinistros,  0)              AS custo_sinistros,
    COALESCE(s.qtd_eventos,      0)              AS qtd_eventos
  FROM receita_mensal         r
  FULL OUTER JOIN sinistros_por_mes s
    ON  r.empresa_id = s.empresa_id
    AND r.ano_mes    = s.ano_mes
)

SELECT
  -- ─ identificação (PK composta) ─────────────────────────────────────
  p.empresa_id,
  p.ano_mes,
  -- ─ atributos da empresa (desnormalizados para consumo direto em BI) ──
  e.empresa_nome,
  e.setor,
  e.porte,
  e.uf,
  -- ─ granularidade temporal ─────────────────────────────────────
  CAST(LEFT(p.ano_mes,  4) AS INT)                                      AS ano,
  CAST(RIGHT(p.ano_mes, 2) AS INT)                                      AS mes,
  -- ─ métricas financeiras ─────────────────────────────────────
  p.receita_premios,
  p.custo_sinistros,
  ROUND(p.receita_premios - p.custo_sinistros, 2)                       AS margem_tecnica,
  -- sinistralidade: ratio (4 casas) e percentual (2 casas)
  CASE
    WHEN p.receita_premios > 0
    THEN ROUND(p.custo_sinistros / p.receita_premios, 4)
    ELSE NULL
  END                                                                    AS sinistralidade,
  CASE
    WHEN p.receita_premios > 0
    THEN ROUND(p.custo_sinistros / p.receita_premios * 100, 2)
    ELSE NULL
  END                                                                    AS sinistralidade_pct,
  -- ─ métricas de volume ──────────────────────────────────────
  p.vidas_ativas,
  p.contratos_ativos,
  p.qtd_eventos,
  -- ticket médio por evento
  CASE WHEN p.qtd_eventos  > 0
    THEN ROUND(p.custo_sinistros / p.qtd_eventos,  2) ELSE 0
  END                                                                    AS ticket_medio_evento,
  -- custo per capita (custo por vida no mês)
  CASE WHEN p.vidas_ativas > 0
    THEN ROUND(p.custo_sinistros / p.vidas_ativas, 2) ELSE 0
  END                                                                    AS custo_per_capita,
  -- frequência de utilização (eventos por vida no mês)
  CASE WHEN p.vidas_ativas > 0
    THEN ROUND(CAST(p.qtd_eventos AS DOUBLE) / p.vidas_ativas, 4) ELSE 0
  END                                                                    AS frequencia_utilizacao,
  -- ─ flag de qualidade [P4] ─────────────────────────────────────
  (p.receita_premios = 0 AND p.custo_sinistros > 0)                     AS sinistro_sem_receita
FROM periodos p
LEFT JOIN homologacao.salutar_saude.gold_dim_empresa e
  ON p.empresa_id = e.empresa_id
ORDER BY p.empresa_id, p.ano_mes;

-- ── Comentários de colunas ─────────────────────────────────────────────────────
ALTER TABLE homologacao.salutar_saude.gold_fat_sinistralidade ALTER COLUMN empresa_id         COMMENT 'FK → gold_dim_empresa | componente da PK';
ALTER TABLE homologacao.salutar_saude.gold_fat_sinistralidade ALTER COLUMN ano_mes             COMMENT 'Período YYYY-MM | componente da PK';
ALTER TABLE homologacao.salutar_saude.gold_fat_sinistralidade ALTER COLUMN receita_premios     COMMENT '[P1] Soma de valor_mensal dos contratos ativos no mês (sem pro-rata, contratos NULL excluídos)';
ALTER TABLE homologacao.salutar_saude.gold_fat_sinistralidade ALTER COLUMN custo_sinistros     COMMENT '[P2] Soma de valor_sinistro por data_evento no mês (sem IBNR)';
ALTER TABLE homologacao.salutar_saude.gold_fat_sinistralidade ALTER COLUMN margem_tecnica      COMMENT 'receita_premios - custo_sinistros. Positivo = margem favorável';
ALTER TABLE homologacao.salutar_saude.gold_fat_sinistralidade ALTER COLUMN sinistralidade      COMMENT 'custo_sinistros / receita_premios (ratio, 4 casas decimais). NULL quando receita = 0 [P4]';
ALTER TABLE homologacao.salutar_saude.gold_fat_sinistralidade ALTER COLUMN sinistralidade_pct  COMMENT 'sinistralidade * 100 (percentual, 2 casas). NULL quando receita = 0 [P4]';
ALTER TABLE homologacao.salutar_saude.gold_fat_sinistralidade ALTER COLUMN vidas_ativas        COMMENT 'Total de vidas nos contratos ativos no mês';
ALTER TABLE homologacao.salutar_saude.gold_fat_sinistralidade ALTER COLUMN ticket_medio_evento COMMENT 'custo_sinistros / qtd_eventos (R$ por evento)';
ALTER TABLE homologacao.salutar_saude.gold_fat_sinistralidade ALTER COLUMN custo_per_capita    COMMENT 'custo_sinistros / vidas_ativas (R$ por vida no mês)';
ALTER TABLE homologacao.salutar_saude.gold_fat_sinistralidade ALTER COLUMN frequencia_utilizacao COMMENT 'qtd_eventos / vidas_ativas (eventos por vida no mês)';
ALTER TABLE homologacao.salutar_saude.gold_fat_sinistralidade ALTER COLUMN sinistro_sem_receita COMMENT '[P4] TRUE = sinistro registrado mas sem receita correspondente; requer investigação';

-- ── Preview: top 10 meses com maior sinistralidade ───────────────────────────
SELECT
  empresa_nome, ano_mes, receita_premios,
  custo_sinistros, sinistralidade_pct,
  margem_tecnica, vidas_ativas, qtd_eventos
FROM homologacao.salutar_saude.gold_fat_sinistralidade
WHERE sinistralidade IS NOT NULL
ORDER BY sinistralidade DESC
LIMIT 10;

empresa_nome,ano_mes,receita_premios,custo_sinistros,sinistralidade_pct,margem_tecnica,vidas_ativas,qtd_eventos
Empresa D04 Ltda,2025-02,12780.03,454859.15,3559.14,-442079.12,17,17
Empresa K37 Ltda,2025-06,16782.14,397252.16,2367.11,-380470.02,40,16
Empresa G07 Ltda,2025-05,14886.11,325488.57,2186.53,-310602.46,17,15
Empresa K37 Ltda,2025-04,16782.14,279009.48,1662.54,-262227.34,40,19
Empresa M13 Ltda,2025-02,24152.20,369512.55,1529.93,-345360.35,107,17
Empresa K37 Ltda,2025-05,16782.14,229202.23,1365.75,-212420.09,40,17
Empresa K37 Ltda,2025-02,16782.14,211461.52,1260.04,-194679.38,40,6
Empresa H34 Ltda,2025-09,75218.43,778612.39,1035.14,-703393.96,328,22
Empresa K37 Ltda,2025-01,16782.14,159474.97,950.27,-142692.83,40,5
Empresa K37 Ltda,2025-03,16782.14,155046.24,923.88,-138264.10,40,9


In [0]:
from pyspark.sql import Row

GOLD_TABLES = [
    "gold_dim_operadora",
    "gold_dim_empresa",
    "gold_dim_plano",
    "gold_dim_corretor",
    "gold_dim_data",
    "gold_dim_beneficiario",
    "gold_fat_contratos",
    "gold_fat_utilizacao",
    "gold_fat_sinistralidade",
]

SCHEMA_PATH = "homologacao.salutar_saude"

print(f"{'Tabela':<30} {'Registros':>12} {'Status'}")
print("-" * 55)
for t in GOLD_TABLES:
    fqn  = f"{SCHEMA_PATH}.{t}"
    n    = spark.table(fqn).count()
    tag  = "[OK]" if n > 0 else "[WARN] VAZIA!"
    print(f"{t:<30} {n:>12,} {tag}")

print("-" * 55)
print(f"[OK] Concluído em {datetime.now():%Y-%m-%d %H:%M:%S}")

Tabela                            Registros Status
-------------------------------------------------------
gold_dim_operadora                        8 [OK]
gold_dim_empresa                         40 [OK]
gold_dim_plano                           28 [OK]
gold_dim_corretor                        25 [OK]
gold_dim_data                         4,748 [OK]
gold_dim_beneficiario                 1,558 [OK]
gold_fat_contratos                      108 [OK]
gold_fat_utilizacao                   6,450 [OK]
gold_fat_sinistralidade                 969 [OK]
-------------------------------------------------------
[OK] Concluído em 2026-07-04 20:58:08
